# Intro
This is human-written with the help of AI, and documenting a learning workflow

## Basics: Understand cURL
A line with ! will be executed as bash (Linux command line)  
cURL is a command line tool that can download from the internet. Test it on a simple website, it will output the source code:

In [ ]:
! curl https://www.cassidae.uni.wroc.pl/katalog%20internetowy/cassidopsis.htm

As you can see, the output of cURL is simply the text file that is at the target of the URL.  
## Understand the TaxonWorks API
Now lets get familiar with the TaxonWorks API.  
Open the following URL with your browser:  
"https://sfg.taxonworks.org/api/v1/taxon_names?name=Curculionoidea&project_token=Ots0-yen4dVefn0Etyxvgw"  
What you'll see is a JSON file, text in a structured format.  
How it is displayed will vary based on your browser. It may be plain text, or more readable in the form of a table.

Every part in the URL has a meaning:

| Part | Meaning |
|---|---|
| `https://sfg.taxonworks.org/api/v1` | **base URL** |
| `/taxon_names` | the **resource** |
| `?` | everything behind the ? is the **query string** |
| `name=Curculionoidea` | A **search parameter**: syntax is key=value|
| `&` | multiple search parameters are separated by & |
| 'project_token=Ots0-yen4dVefn0Etyxvgw | This search parameter or key=value pair is the **project token**, its like a password

Now test the request with curl:


In [ ]:
! curl "https://sfg.taxonworks.org/api/v1/taxon_names?name=Curculionoidea&project_token=Ots0-yen4dVefn0Etyxvgw"

Search parameters can be given in any order. Here, project_token and name are swapped, output should be the same:

In [ ]:
! curl "https://sfg.taxonworks.org/api/v1/taxon_names?&project_token=Ots0-yen4dVefn0Etyxvgw&name=Curculionoidea"

Plain JSON is difficult to read. Pipe the python script json.tool to have more readable output:

In [ ]:
 ! curl "https://sfg.taxonworks.org/api/v1/taxon_names?&project_token=Ots0-yen4dVefn0Etyxvgw&name=Curculionoidea" | python3 -m json.tool

Or, to work directly in Python:

In [ ]:
import requests, json

r = requests.get(
    "https://sfg.taxonworks.org/api/v1/taxon_names",
    params={"name": "Curculionoidea", "project_token": "Ots0-yen4dVefn0Etyxvgw"}
)
# What is the result of requests.get? remove the comment #  to see
#print(r)
#print(r.status_code)   # 200, 404, etc. — same info __repr__ shows
#print(r.url)            # the fully-built URL, useful for debugging query params
#print(r.text)           # raw response body as a string
#print(r.json())         # response body parsed into a Python dict/list (what you called `data`)
#print(r.headers)        # response headers, e.g. content-type

To show in a more readable format:

In [ ]:
import requests, json

r = requests.get(
    "https://sfg.taxonworks.org/api/v1/taxon_names",
    params={"name": "Curculionoidea", "project_token": "Ots0-yen4dVefn0Etyxvgw"}
)
# or to format nicely:
print(json.dumps(
    r.json(),
    indent=2)
     )

this can be turned into a handy helper function:

In [ ]:
import requests, json
# defining a function:
def showJSON(request):
    print(json.dumps(request.json(), indent=2))
    

r = requests.get(
    "https://sfg.taxonworks.org/api/v1/taxon_names",
    params={"name": "Curculionoidea", "project_token": "Ots0-yen4dVefn0Etyxvgw"}
)

showJSON(r)

The format of the response depends on the shape of r.json(), for taxon_names, it's a JSON array of objects, so you get a **Python list of dicts**.  
A few ways to pull elements out:

In [ ]:
import requests, json

r = requests.get(
    "https://sfg.taxonworks.org/api/v1/taxon_names",
    params={"name": "Curculionoidea", "project_token": "Ots0-yen4dVefn0Etyxvgw"}
)

data = r.json()

#len(data)          # how many results came back
data[0]             # the first result (a dict)
#data[0]["rank"]     # a specific field of the first result

Try a different resource:

In [ ]:
import requests, json
# defining a function:
def showJSON(request):
    print(json.dumps(request.json(), indent=2))
    

r = requests.get(
    "https://sfg.taxonworks.org/api/v1/dwc_occurrences",
    params={
    "project_token": "Ots0-yen4dVefn0Etyxvgw"
    }
)

showJSON(r)

In [ ]:
import requests, json

r = requests.get(
    "https://sfg.taxonworks.org/api/v1/dwc_occurrences",
    params={
    "project_token": "Ots0-yen4dVefn0Etyxvgw",
    "country": "Germany"
    }
)

data = r.json()

len(data)

In [ ]:
import requests, json

r = requests.get(
    "https://sfg.taxonworks.org/api/v1/otus/732685/inventory/dwc.json",
    params={
    "project_token": "Ots0-yen4dVefn0Etyxvgw",
    "country": "Germany"
    }
)

data = r.json()

print (data)

In [ ]:
import requests, json
def showJSON(request):
    print(json.dumps(request.json(), indent=2))

r = requests.get(
    "https://sfg.taxonworks.org/api/v1/dwc_occurrences",
    params={
    "project_token": "Ots0-yen4dVefn0Etyxvgw",
    "taxon_name_id": 809411,
    "country": "Germany",
    "descendants": "true"
    }
)

data = r.json()
len(data)


## Fetch country status for all weevil families
To do: replace taxon_names resource with: /otus/<otu_id>/inventory/dwc.json

      "project_token": TOKEN,
      "family": "Curculionidae",   # or genus=/tribe=
      "country": country,
      "occurrenceStatus": "present",   # ← excludes explicit absence statements
      "per": 1,
  }

In [ ]:
import requests, json
# tuple with hard-coded country names
scopetaxon_id = 809411
Adosomus = 833911
countries = ("Germany", "Austria", "Georgia")

def showJSON(request):
    print(json.dumps(request.json(), indent=2))

# get direct descendants of Curculionoidea
descendants = requests.get(
    "https://sfg.taxonworks.org/api/v1/taxon_names",
    params={
    "project_token": "Ots0-yen4dVefn0Etyxvgw",
    "parent_id[]": Adosomus,
    }
)
children = descendants.json()

results = [] # define results as a list
for child in children:
    child_name = child["name"] # id is a python function, needs to be string in quotes
    child_rank = child["rank"]
    for country in countries:
         r = requests.get(
         "https://sfg.taxonworks.org/api/v1/dwc_occurrences",
         params={
         "project_token": "Ots0-yen4dVefn0Etyxvgw",
         child_rank: child_name, # "family": "Curculionidae",
         "country": country,
         "per": "1"
         },
    )
    data = r.json()
    record = {"taxon": child_name, "country": country, "data": data}
    if len(data) > 0:
        results.append(record)

# families is a list of dicts. 
# iterate through the list, for the taxon name id of each line, run the following:


print(results)

## Sandbox

In [ ]:
Documented in docs/feasibility_key_geography_filter.md. Here's the winning request, one more time, clean:

  GET /dwc_occurrences
    ?family=<Name>              (or subfamily=, genus=, tribe= — match the terminal's actual rank)
    &country=<CountryName>
    &occurrenceStatus=present   (required — excludes explicit "not found here" statements)
    &per=1

  Read presence from the pagination-total response header (not response body length, which is capped by per).

  The comparison, restated:

  ┌────────────────────────────────────────────────────────────────────┬───────────────────────────────────────────────────────────┐
  │                                                                    │ Time (key 5024, higher-rank terminals, all 197 countries) │
  ├────────────────────────────────────────────────────────────────────┼───────────────────────────────────────────────────────────┤
  │ Current shipped method (asserted_distributions + descendants=true) │ 62.2s                                                     │
  ├────────────────────────────────────────────────────────────────────┼───────────────────────────────────────────────────────────┤
  │ This method (dwc_occurrences flat columns)                         │ 31.4–36s                                                  │
  └────────────────────────────────────────────────────────────────────┴───────────────────────────────────────────────────────────┘

  And the "sees more data" part isn't a separate feature to add — it's a side effect of what the cache already is. dwc_occurrences is built as a union of three source tables (asserted distributions, collection
  objects, field occurrences), so a single family=Curculionidae&country=Germany probe already returned a mix — confirmed live: {'AssertedDistribution': 47, 'CollectionObject': 3} — in one request. The current
  shipped code only gets that mix for species/small-genus terminals; it deliberately skips the specimen half at family/tribe rank because the naive way to fetch it (inventory/dwc.json, no country filter) is a
  100MB+ call there. This method doesn't have that cost, so it doesn't need that skip.
